# Marketing Funnel & Conversion Analysis
**Future Interns — Data Science & Analytics · Task 3 · 2026**  
**Author:** Daksh Singh · Zip Innovate Technology  
**Dataset:** Bank Marketing Campaign · UCI ML Repository · 45,211 records

---
### Notebook Outline
1. Load & inspect dataset
2. Exploratory data analysis
3. Funnel stage mapping
4. Channel performance
5. Monthly trends
6. Key predictor analysis (duration, job, age)
7. Insights & recommendations


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', None)
print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/bank_marketing.csv')
print(f'Shape: {df.shape}')
print(f'Subscription rate: {(df["y"]=="yes").mean()*100:.2f}%')
df.head()

In [ ]:
df.info()
print('\nNull values:')
print(df.isnull().sum())

In [ ]:
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
# Target distribution
print(df['y'].value_counts())
print(df['y'].value_counts(normalize=True).mul(100).round(2))

In [ ]:
fig = px.histogram(df, x='age', color='y', barmode='overlay',
                   title='Age Distribution by Subscription Outcome',
                   nbins=30, color_discrete_map={'yes':'#00e5ff','no':'#f43f5e'})
fig.show()

In [ ]:
# Job vs subscription rate
job_sub = df.groupby('job')['y'].apply(lambda x: (x=='yes').mean()*100).reset_index()
job_sub.columns = ['job','sub_rate']
job_sub = job_sub.sort_values('sub_rate', ascending=False)
print(job_sub.to_string(index=False))

In [ ]:
fig = px.bar(job_sub.sort_values('sub_rate'), y='job', x='sub_rate',
             orientation='h', title='Subscription Rate by Job Category (%)',
             color='sub_rate', color_continuous_scale='Teal')
fig.show()

## 3. Funnel Stage Mapping

In [ ]:
MONTH_ORDER = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
CHANNEL_MAP = {'cellular':'Mobile / Cellular','telephone':'Landline / Telephone','unknown':'Unknown / Direct'}

df['channel']     = df['contact'].map(CHANNEL_MAP).fillna('Other')
df['is_visitor']  = 1
df['is_lead']     = (df['duration'] > 0).astype(int)
df['is_mql']      = ((df['campaign'] > 1) | (df['previous'] > 0)).astype(int)
df['is_sql']      = ((df['pdays'] != -1) | (df['poutcome'] == 'success')).astype(int)
df['is_customer'] = (df['y'] == 'yes').astype(int)

stages = {'Contacted': df['is_visitor'].sum(), 'Leads': df['is_lead'].sum(),
          'MQLs': df['is_mql'].sum(), 'SQLs': df['is_sql'].sum(),
          'Subscribers': df['is_customer'].sum()}

print('=== FUNNEL SUMMARY ===')
base = stages['Contacted']
for stage, count in stages.items():
    print(f'{stage:12s}: {count:>6,}  ({count/base*100:.1f}% of contacted)')

In [ ]:
fig = go.Figure(go.Funnel(
    y=list(stages.keys()), x=list(stages.values()),
    texttemplate='<b>%{label}</b><br>%{value:,.0f} (%{percentInitial:.1%})',
    marker=dict(color=['#00e5ff','#38b2f9','#7c3aed','#a855f7','#f59e0b'])
))
fig.update_layout(title='Bank Marketing Conversion Funnel', height=420)
fig.show()

## 4. Channel Performance

In [ ]:
ch = df.groupby('channel')[['is_visitor','is_lead','is_mql','is_sql','is_customer']].sum().reset_index()
ch.columns = ['channel','visitors','leads','mqls','sqls','customers']
ch['cvr'] = (ch['customers'] / ch['visitors'] * 100).round(3)
print(ch.to_string(index=False))

In [ ]:
fig = px.bar(ch.sort_values('cvr', ascending=False), x='channel', y='cvr',
             color='channel', title='CVR % by Contact Channel',
             text=ch.sort_values('cvr',ascending=False)['cvr'].apply(lambda v:f'{v:.2f}%'),
             color_discrete_sequence=['#00e5ff','#7c3aed','#f59e0b'])
fig.update_traces(textposition='outside')
fig.show()

## 5. Monthly Trends

In [ ]:
monthly = df.groupby('month')[['is_visitor','is_lead','is_customer']].sum().reset_index()
monthly.columns = ['month','contacted','leads','subscribers']
monthly['month_order'] = monthly['month'].apply(lambda m: MONTH_ORDER.index(m.lower()) if m.lower() in MONTH_ORDER else 99)
monthly = monthly.sort_values('month_order')
monthly['cvr'] = (monthly['subscribers'] / monthly['contacted'] * 100).round(3)
print(monthly[['month','contacted','leads','subscribers','cvr']].to_string(index=False))

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['contacted'],
                         name='Contacted', fill='tozeroy', line=dict(color='#00e5ff')), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly['month'], y=monthly['subscribers'],
                         name='Subscribers', line=dict(color='#f59e0b'), mode='lines+markers'), secondary_y=True)
fig.update_layout(title='Monthly Contacted vs Subscribers', height=350)
fig.show()

## 6. Call Duration Analysis

In [ ]:
print('Avg call duration:')
print(df.groupby('y')['duration'].agg(['mean','median','std']).round(1))

fig = px.box(df, x='y', y=df['duration'].clip(upper=2000),
             color='y', title='Call Duration by Subscription Outcome',
             color_discrete_map={'yes':'#00e5ff','no':'#f43f5e'},
             labels={'y':'Subscribed','duration':'Duration (sec)'})
fig.show()

## 7. Key Insights

In [ ]:
overall_cvr  = df['is_customer'].mean()*100
v2l          = df['is_lead'].mean()*100
best_ch      = ch.loc[ch['cvr'].idxmax()]
best_job     = df.groupby('job')['is_customer'].mean().idxmax()
avg_dur_yes  = df[df['y']=='yes']['duration'].mean()
avg_dur_no   = df[df['y']=='no']['duration'].mean()

print('=== KEY INSIGHTS ===')
print(f'Overall CVR (Contact→Subscriber) : {overall_cvr:.2f}%')
print(f'Contact→Lead Rate                : {v2l:.1f}%')
print(f'Best channel                     : {best_ch["channel"]} ({best_ch["cvr"]:.2f}% CVR)')
print(f'Best job segment                 : {best_job}')
print(f'Avg call duration (subscribed)   : {avg_dur_yes:.0f}s')
print(f'Avg call duration (not subscribed): {avg_dur_no:.0f}s')
print()
print('RECOMMENDATIONS')
print('1. Train agents to extend call duration — strongest single predictor of subscription.')
print('2. Shift budget to cellular/mobile contacts — highest CVR channel.')
print('3. Build re-contact sequences for previous contacts (previous > 0).')
print(f'4. Prioritize "{best_job}" segment — highest base subscription rate.')
print('5. Pre-load May campaigns 6 weeks early — peak volume month.')